# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** CTR / Engagement Opportunity Scoring. **Question shape:** "which visible pages should an editor review first?", which is ranking, judged at Precision@K.

**Method:** a classifier's probability of `under_captures_next` (the page under-clicks for its position in the next two weeks), used as a ranking score. I compare three models of rising complexity against my Week-4 rule:

| Method | Why it's on the list |
|---|---|
| **Week-4 baseline** (clicks-at-stake rule) | the bar to beat: transparent, no fitted weights |
| **Logistic Regression** | readable: one weight per feature, tells me if a simple linear mix is enough |
| **Random Forest** | captures interactions (position × volume × content type) without much tuning |
| **Gradient Boosting** (HistGradientBoosting) | usually the strongest on tabular data; "safe" here because every input is from the feature window |

Complexity only counts if it beats the rule **on unseen clients**. If Logistic Regression ties the forest, I prefer Logistic Regression.

**Features (all from 1–15 March, from my ML-04 contract plus two derived from the same window):** `log impr_f`, `ctr_f`, `pos_f`, `content_type`, `age_days`, plus `expected_ctr` (the bucket median of `ctr_f`) and `ctr_ratio_f` (= `ctr_f / expected_ctr`, the gap the Week-4 rule is built on). Nothing from 16–31 March and no product flags.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, subprocess, duckdb, numpy as np, pandas as pd, sklearn
from huggingface_hub import hf_hub_download, get_token
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("work") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

SEED = 42
assert get_token(), "No Hugging Face token: log in (hf auth login) or add HF_TOKEN in Colab Secrets"
REPO = "FlyRank/internship-warehouse"
con = duckdb.connect()
for name, path in {"fact_mar": "fact_content_daily_performance/month=2026-03/data_0.parquet",
                   "dim_content": "dim_content.parquet"}.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{hf_hub_download(REPO, path, repo_type='dataset')}')")

DECISION = "2026-03-16"   # identical slice + label to ML-04 / ML-07
frame = con.sql(f"""
    WITH f AS (SELECT client_hash_id, content_hash_id,
                      SUM(gsc_impressions) AS impr_f, SUM(gsc_clicks) AS clicks_f,
                      SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS pos_f
               FROM fact_mar WHERE gsc_data_available IS TRUE AND report_date <  DATE '{DECISION}' GROUP BY 1, 2),
         o AS (SELECT content_hash_id, SUM(gsc_impressions) AS impr_o, SUM(gsc_clicks) AS clicks_o
               FROM fact_mar WHERE gsc_data_available IS TRUE AND report_date >= DATE '{DECISION}' GROUP BY 1)
    SELECT f.client_hash_id, f.content_hash_id, f.impr_f, f.clicks_f,
           100.0 * f.clicks_f / f.impr_f AS ctr_f, f.pos_f, c.content_type, c.main_intent,
           DATE_DIFF('day', c.content_created_date, DATE '{DECISION}') AS age_days,
           100.0 * o.clicks_o / o.impr_o AS ctr_o
    FROM f JOIN o USING (content_hash_id) LEFT JOIN dim_content c USING (content_hash_id)
    WHERE f.impr_f >= 100 AND o.impr_o >= 100 AND f.pos_f BETWEEN 1 AND 20
""").df().sort_values("content_hash_id").reset_index(drop=True)
frame["pos_bucket"] = pd.cut(frame["pos_f"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
frame["under_captures_next"] = (frame["ctr_o"] < 0.5 * frame.groupby("pos_bucket", observed=True)["ctr_o"].transform("median")).astype(int)
frame = frame.drop(columns=["ctr_o"])          # outcome column gone before any modeling
frame["expected_ctr"] = frame.groupby("pos_bucket", observed=True)["ctr_f"].transform("median")
frame["ctr_ratio_f"] = frame["ctr_f"] / frame["expected_ctr"]
# Week-4 baseline score, frozen exactly as built
elig = (frame["impr_f"] >= 500) & (frame["ctr_ratio_f"] < 0.5)
frame["baseline_score"] = np.where(elig, (frame["expected_ctr"] - frame["ctr_f"]) / 100 * frame["impr_f"], 0.0)

print(f"Frame: {len(frame):,} pages, {frame['client_hash_id'].nunique()} clients, base rate {frame['under_captures_next'].mean():.3f}")
print(f"seed={SEED}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}")


Frame: 57,392 pages, 36 clients, base rate 0.398
seed=42, scikit-learn 1.9.1, pandas 3.1.1


## 2. Split design

**Grouped 5-fold cross-validation by client (`GroupKFold` on `client_hash_id`).** In each fold, roughly a fifth of the clients are held out completely: none of their pages are seen in training. Every page is scored exactly once, by a model that never saw its site.

Why this is honest for my question:
- The queue will be used on **sites the model hasn't seen**, and pages from one site share templates, niche and brand, so a random page split would leak site identity and flatter the model (notebook 02 showed the gap shrinking under client holdout).
- **Time is respected inside each row:** features come from 1–15 March and the label from 16–31 March, so no row uses the future. (A second, later decision moment, e.g. April, would be the next step for a time-forward check. Section 4 notes this limit.)
- **5 folds instead of one split**, because with 36 clients a single 20% holdout is only ~7 clients, and ML-01/02 showed one split can tell either story. I report the mean **and** the spread.

**Same candidate pool as the editor:** Precision@50 is computed among pages with **≥500 feature-window impressions**, the same pool the Week-4 rule can flag. Otherwise a model could "win" by surfacing tiny pages (which do persist, ML-07 signal 2) that aren't worth an editor's hour.


In [2]:
groups = frame["client_hash_id"].values
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(frame, groups=groups))
for i, (tr, te) in enumerate(folds):
    pool = te[frame.loc[te, "impr_f"].values >= 500]
    print(f"fold {i}: train {len(tr):>6,} pages / test {len(te):>6,} pages from {len(set(groups[te]))} clients "
          f"| editor pool {len(pool):>5,} | test base rate {frame.loc[te, 'under_captures_next'].mean():.3f}")
# sanity: no client in both train and test
assert all(not set(groups[tr]) & set(groups[te]) for tr, te in folds)


fold 0: train 42,368 pages / test 15,024 pages from 1 clients | editor pool 10,006 | test base rate 0.413
fold 1: train 45,448 pages / test 11,944 pages from 1 clients | editor pool 7,914 | test base rate 0.437
fold 2: train 47,250 pages / test 10,142 pages from 11 clients | editor pool 4,974 | test base rate 0.376
fold 3: train 47,251 pages / test 10,141 pages from 9 clients | editor pool 5,083 | test base rate 0.405
fold 4: train 47,251 pages / test 10,141 pages from 14 clients | editor pool 5,615 | test base rate 0.346


## 3. Train + compare vs my baseline

Every method is scored in the same run, on the same five folds, with the same metric. The **main metric is Precision@50 in the editor pool** (≥500 impressions), with ROC-AUC on all test pages as a secondary check of overall ranking quality.

In [3]:
NUM = ["log_impr_f", "ctr_f", "pos_f", "age_days", "expected_ctr", "ctr_ratio_f"]
frame["log_impr_f"] = np.log1p(frame["impr_f"])
frame["age_days"] = frame["age_days"].fillna(frame["age_days"].median())
X = pd.concat([frame[NUM], pd.get_dummies(frame["content_type"].fillna("unknown"), prefix="type", dtype=float)], axis=1)
FEATS = list(X.columns)
y = frame["under_captures_next"].values

def make_models():
    return {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED),
        "Gradient Boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=SEED),
    }

def p_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

oof = {m: np.zeros(len(frame)) for m in make_models()}
rows = []
for i, (tr, te) in enumerate(folds):
    pool = frame.loc[te, "impr_f"].values >= 500
    scores = {"Week-4 baseline": frame.loc[te, "baseline_score"].values,
              "Random pick (base rate)": np.random.default_rng(SEED + i).random(len(te))}
    for name, model in make_models().items():
        model.fit(X.iloc[tr], y[tr])
        scores[name] = model.predict_proba(X.iloc[te])[:, 1]
        oof[name][te] = scores[name]
    for name, s in scores.items():
        rows.append({"fold": i, "method": name,
                     "P@50 (pool)": p_at_k(s[pool], y[te][pool], 50),
                     "P@20 (pool)": p_at_k(s[pool], y[te][pool], 20),
                     "ROC-AUC (all test)": roc_auc_score(y[te], s)})

res = pd.DataFrame(rows)
order = ["Random pick (base rate)", "Week-4 baseline", "Logistic Regression", "Random Forest", "Gradient Boosting"]
table = (res.groupby("method")
            .agg(**{"P@50 mean": ("P@50 (pool)", "mean"), "P@50 std": ("P@50 (pool)", "std"),
                    "P@20 mean": ("P@20 (pool)", "mean"), "ROC-AUC mean": ("ROC-AUC (all test)", "mean")})
            .reindex(order))
wins = res.pivot(index="fold", columns="method", values="P@50 (pool)")
table["folds beating baseline (P@50)"] = [int((wins[m] > wins["Week-4 baseline"]).sum()) if m != "Week-4 baseline" else "-" for m in order]
print("Model vs baseline - 5 client-grouped folds, same data, same metric:")
print(table.round(3).to_string())
print("\nPer-fold P@50 (editor pool):")
print(wins[order].round(2).to_string())

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w05_model_metrics.json", "w") as fh:
    json.dump({"split": "GroupKFold(5) by client_hash_id", "pool": "impr_f >= 500", "seed": SEED,
               "sklearn": sklearn.__version__,
               "table": table.drop(columns=["folds beating baseline (P@50)"]).round(4).reset_index().to_dict(orient="records")}, fh, indent=2)


Model vs baseline - 5 client-grouped folds, same data, same metric:
                         P@50 mean  P@50 std  P@20 mean  ROC-AUC mean folds beating baseline (P@50)
method                                                                                             
Random pick (base rate)      0.248     0.081       0.33         0.503                             0
Week-4 baseline              0.780     0.014       0.80         0.547                             -
Logistic Regression          0.656     0.065       0.70         0.780                             0
Random Forest                0.780     0.062       0.83         0.807                             2
Gradient Boosting            0.776     0.062       0.82         0.807                             1

Per-fold P@50 (editor pool):
method  Random pick (base rate)  Week-4 baseline  Logistic Regression  Random Forest  Gradient Boosting
fold                                                                                              

### Reading the table

- **At the decision the editor actually makes (top 50 of the ≥500-impression pool), the models do not beat my rule.** Week-4 baseline: **0.780 ± 0.014**. Random Forest: **0.780 ± 0.062**. Gradient Boosting: **0.776 ± 0.062**. Logistic Regression is clearly worse (0.656). The forest beat the rule on only 2 of 5 folds, boosting on 1. The rule is also **four times more stable** across folds (std 0.014 vs 0.062).
- **At P@20 the forest edges ahead (0.83 vs 0.80),** a 1–2 page difference in 20, too small to call.
- **Across the whole list, the models are much better: ROC-AUC 0.81 vs 0.55.** That's partly by construction: the rule gives a score of 0 to every page outside its flag, so it can't order the bottom of the list at all. But it does mean the model ranks *all* pages sensibly, for example for a deeper queue (top 500) or for monitoring.
- **Random pick in the pool is 0.25**, below the overall base rate of 0.40, because big pages under-capture less often than small ones (ML-07, signal 2). Both the rule and the models are ~3× better than random.
- **Caveat on the split:** `GroupKFold` balances page counts, so folds 0 and 1 each hold out **one very large client** (15k and 12k pages). Those two folds test "generalise to one big unseen site"; folds 2–4 test "many smaller unseen sites". Results are similar on both, which is reassuring, but 36 clients is still few.

**Decision: complexity isn't rewarded here.** For the editor's top-50 queue I keep the **Week-4 rule** as the primary ranking: same precision, more stable, explainable in one sentence. The Random Forest is worth keeping only as a *second opinion* deeper in the list, where the rule has nothing to say.

## 4. Errors and interpretation

**What the model leans on (permutation importance, Gradient Boosting, held-out fold 0):**
1. **`log_impr_f`** (ROC-AUC drop 0.11): by far the biggest. Lower-impression pages are more likely to keep under-capturing (consistent with ML-07's volume finding). Plausible: small pages often rank for long-tail queries where a weak snippet stays weak, but it also means the model partly predicts *noise persistence* on small pages.
2. **`ctr_ratio_f`** (0.05) and **3. `ctr_f`** (0.04): the CTR-vs-position gap, the same signal my rule is built on. So the model mostly rediscovered the rule's signal and added volume on top.
- `pos_f`, `age_days` and content type barely matter; `expected_ctr` adds nothing beyond `pos_f`. Nothing looks "suspiciously perfect", so there's no sign of leakage (the deliberate-leak run in ML-04 hit 1.000; this is 0.81).

**Where it's wrong (top-decile flags, out-of-fold):** precision is fairly even, 0.72–0.77 across position buckets and intents. The weakest groups are **positions 4–10 (0.72)** and **transactional intent (0.70)**, plausibly because those pages' CTR depends on ads and shopping results above them, which the data doesn't contain. Comparison articles look great (0.90) but n = 165, and ML-01 found they come from a single client, so I don't read anything into it.

**Three confident misses** (scored ~0.84 but did fine next window): all are keyword articles with **very large impressions (5k–16k) and near-zero CTR (0.006–0.04%)**. They look exactly like the rule's weak picks in ML-07: a page showing up for a broad query it doesn't really answer. Its CTR can jump around on a few clicks when the query mix shifts week to week. They're hard because nothing in five aggregate features says *which queries* drove the impressions; the query table (`fact_content_query_90d`) could add query concentration in the capstone.

**Limits of this result:** one decision moment (16 March), a two-week outcome window, 36 clients, and a label that's a proxy for "keeps under-clicking", not "would gain clicks after a rewrite". The next honest step is a **time-forward check**: freeze the rule and the model on March and score them on a later month. That's ML-09's job.


In [4]:
# (a) What does the model lean on? Permutation importance on held-out fold 0 (drop in ROC-AUC when a feature is shuffled)
best = "Gradient Boosting"
tr, te = folds[0]
m = make_models()[best].fit(X.iloc[tr], y[tr])
pi = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
print(f"Permutation importance ({best}, fold 0, ROC-AUC drop):")
print(imp.head(8).round(4).to_string())

# (b) Where is it most wrong? Out-of-fold error rate of the top-scored decile, by group
d = frame.assign(score=oof[best])
d["flagged"] = d["score"] >= d["score"].quantile(0.9)
fp = d[d["flagged"]]
print(f"\nTop-decile flags: {len(fp):,} pages, precision {fp['under_captures_next'].mean():.3f}")
for col in ["pos_bucket", "content_type", "main_intent"]:
    g = fp.groupby(col, observed=True)["under_captures_next"].agg(n="size", precision="mean")
    print(f"\nPrecision of top-decile flags by {col}:")
    print(g[g["n"] >= 30].round(3).to_string())

# (c) Three concrete wrong cases: highest-scored pool pages that did NOT under-capture
wrong = d[(d["impr_f"] >= 500) & (d["under_captures_next"] == 0)].nlargest(3, "score")
print("\nThree confident misses (high score, page did fine next window):")
print(wrong[["content_type", "main_intent", "pos_f", "impr_f", "ctr_f", "expected_ctr", "ctr_ratio_f", "score"]].round(3).to_string(index=False))


Permutation importance (Gradient Boosting, fold 0, ROC-AUC drop):
log_impr_f                 0.1125
ctr_ratio_f                0.0496
ctr_f                      0.0435
age_days                   0.0046
pos_f                      0.0043
expected_ctr               0.0000
type_comparison article    0.0000
type_feedly article        0.0000

Top-decile flags: 5,740 pages, precision 0.747

Precision of top-decile flags by pos_bucket:
               n  precision
pos_bucket                 
1-3          517      0.774
4-10        2566      0.720
11-20       2657      0.769

Precision of top-decile flags by content_type:
                       n  precision
content_type                       
comparison article   165      0.903
feedly article        91      0.670
keyword article     5484      0.744

Precision of top-decile flags by main_intent:
                  n  precision
main_intent                   
commercial     1025      0.763
informational  3485      0.755
navigational     35      0.77

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.